<!-- cabecera-entorno -->
## Antes de empezar

**Clase 13 · Estadística inferencial** — Bloque 2 · Demo. Este cuaderno se recorre **por su cuenta**:
explica cada concepto antes de usarlo, y el profesor circula por el salón resolviendo dudas. No hay
que esperar a que alguien lo dicte.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/educacion_estadisticas.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 13 · Demo — Estadística inferencial

**Dataset:** `../datos/educacion_estadisticas.csv`, indicadores de educación preescolar, básica
y media del Ministerio de Educación Nacional, vía datos.gov.co. 482 filas x 37 columnas, un registro
por departamento y año, entre 2011 y 2024.

## Cómo se usa este cuaderno

Está escrito para que usted avance solo. Cada bloque de código viene precedido de la explicación del
concepto que usa, y cada término nuevo se define la primera vez que aparece. **Hay que leer antes de
ejecutar.**

**El recorrido:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 y 1 | El dataset, las librerías, la carga, la limpieza y su verificación | El dato en memoria y la certeza de que está bien |
| 2 | Qué es una muestra y por qué no es la población. **Simulado** | La distribución muestral y el error estándar |
| 3 | El intervalo de confianza: la fórmula y las dos palancas que lo mueven | Saber construirlo |
| 4 | Qué significa el 95%, y qué **no**. **Simulado, 200 estudios** | El punto donde casi todo el mundo se equivoca |
| 5 | Los tres errores que no avisan | Saber cuándo desconfiar de un número que salió limpio |
| 6 | El tribunal: H0, H1 y los 4 pasos. Qué es un p-valor. **Simulado, 2.000 pruebas** | La definición y sus tres malinterpretaciones |
| 7 | Prueba de una muestra contra una meta | `stats.ttest_1samp` |
| 8 | Prueba de dos muestras: urbanizado contra rural disperso | `stats.ttest_ind` y el IC de la diferencia |
| 9 | Por qué "significativo" no quiere decir "importante". **Simulado** | La pregunta que salva un informe |
| 10 | Cómo se reporta un resultado | La plantilla que se usa el resto del semestre |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Escribir código es el bloque 3, con el reto, y es
lo que se entrega.

**Las quince preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.** Abrirlo
antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación es que usted sepa mirar un
resultado y decir qué significa, no que sepa reproducir un cálculo. Y en esta clase, más que en
ninguna, el número correcto con la frase equivocada está **mal**.

**Las secciones "Para entender qué está pasando"** explican el fundamento: qué es realmente una
muestra, qué significa el 95%, qué mide un p-valor. Son las que convierten esto en análisis en vez de
recetas. Si ya las tiene claras, se pueden saltar sin perder el hilo del código.

**Y una advertencia que vale para toda la clase de hoy:** esta es la clase donde el código es corto y
la interpretación es larga. Son cuatro llamadas a `scipy` en total. Un cuaderno lleno de números
correctos con una conclusión mal redactada está **mal**, y ningún intérprete de Python se lo va a
decir. Por eso la mitad de este cuaderno son simulaciones: los conceptos de hoy no se creen, se ven.

---

### Lo que este cuaderno da por sabido

No se repite aquí nada de lo que ya se explicó. Si algo de esta lista no le suena, vuelva al cuaderno
que lo enseña **antes** de seguir.

| Lo que se da por sabido | Dónde se explicó |
|-------------------------|------------------|
| Librería, alias, DataFrame, Series, índice, dtype, `read_csv`, `head`, `describe` | Clase 2, demo, secciones 1 a 5 |
| Máscara booleana, `.isin()`, filtrado | Clase 2, demo, secciones 7 a 11 |
| Nulos, `NaN`, `dropna`, conversión de tipos, duplicados, valores imposibles de dominio | Clase 3, demo, pasos 1 a 6 |
| Media, mediana, **desviación estándar**, histograma, boxplot | Clase 4, demo, secciones 3, 5 y 6 |
| `groupby` y comparación de medias entre categorías | Clase 4, demo, sección 4, y clase 5, demo, sección 5 |
| Correlación no es causalidad, y las tres explicaciones de toda asociación | Clase 5, demo, sección 8 |

**Lo nuevo de hoy** es el salto de describir a **afirmar**: la distribución muestral, el error estándar,
el intervalo de confianza, la prueba de hipótesis y el p-valor. Eso sí va explicado desde cero.

La frase que resume el salto, y que conviene copiar:

> La estadística **descriptiva** describe lo que midió. La **inferencial** afirma algo sobre lo que
> no midió, y le pone un número a lo que puede equivocarse.

---

### El marco de 4 pasos de una prueba de hipótesis

Es el esqueleto de la segunda mitad de la clase, y se usa idéntico en el reto y en el Momento 3.

| Paso | Nombre | Qué se hace |
|------|--------|-------------|
| 1 | Plantear | H0 y H1, en español, **antes** de mirar los datos |
| 2 | Elegir alfa | 0.05 salvo que haya razón para otra cosa |
| 3 | Calcular | Descriptivos primero, después el estadístico y el p-valor |
| 4 | Decidir y redactar | p < alfa, se rechaza H0. Y se escribe la conclusión en lenguaje de negocio |

**El paso 1 va antes que el 3 a propósito.** Escribir H0 después de ver el p-valor no es un descuido de
método: es el mecanismo por el que se fabrican hallazgos que no existen.

---

## 0. El dataset, antes de tocarlo

Regla de la casa: nunca se ejecuta una línea de código sobre un dataset que no se sabe qué es.

| Campo | Valor |
|-------|-------|
| Qué mide | Indicadores de educación preescolar, básica y media |
| Fuente | Ministerio de Educación Nacional, publicado en datos.gov.co |
| Tamaño | 482 filas x 37 columnas antes de limpiar |
| Granularidad | Una fila = un departamento, un año |
| Cobertura | 2011 a 2024, 33 departamentos |

**Las columnas de hoy:**

| Columna | Qué mide | Unidad |
|---------|----------|--------|
| `ano` | Año del registro | año |
| `c_digo_departamento` | Código DANE del departamento | entero |
| `tasa_matriculacion_5_16` | Porcentaje de la población de 5 a 16 años matriculada | % |
| `desercion` | Porcentaje de estudiantes que abandonan el año | % |

**Qué tiene de sucio**, verificado columna por columna:

| Problema | Evidencia | Qué se hace |
|----------|-----------|-------------|
| `ano` llega como decimal | Hay `2018.0` conviviendo con `2018` | `astype(int)` |
| `departamento` inconsistente | `'  Nariño  '`, `'antioquia'`, `'Guainia'` y `'Guainía'`, `'BOGOTA, D,C,'` y `'BOGOTA, D.C.'` conviven | Se estandariza en la carga con la receta de la clase 3. Y aun así **la llave es `c_digo_departamento`**: un código no tiene ortografía |
| `poblacion_5_16` llega como texto | Valores tipo `"394,574"`, con coma de miles | `str.replace(',', '')` y `pd.to_numeric` |
| Duplicados | 20 pares (año, departamento) repetidos | `drop_duplicates`, quedan 462 filas |
| Nulos dispersos | `desercion` 47, `aprobacion` 55, `cobertura_neta` 40 | `dropna()` al momento del cálculo, y **se reporta el n resultante** |
| `tamano_promedio_grupo` inservible | Media de 6.162 para un "tamaño promedio de grupo" | Se descarta la columna, y se dice por qué |

**Y la regla de honestidad que rige toda la clase:** si el `n` que usted reporta no coincide con las
filas del archivo, eso no está mal, está **bien reportado**. Lo que está mal es no decirlo.

---

## 1. Las librerías de hoy

**pandas** (`pd`) y **numpy** (`np`) ya los conoce. **matplotlib** (`plt`) también.

**scipy** es la librería nueva, y de ella solo usamos un submódulo: `scipy.stats`, que trae las
funciones estadísticas. Son cuatro en toda la clase:

| Función | Qué hace |
|---------|----------|
| `stats.sem(x)` | El error estándar de la media de `x` |
| `stats.t.interval(...)` | El intervalo de confianza de una media |
| `stats.ttest_1samp(x, popmean=...)` | Compara la media de un grupo contra un número de referencia |
| `stats.ttest_ind(a, b, equal_var=False)` | Compara las medias de dos grupos independientes |

Cuatro llamadas. Todo lo demás de hoy es interpretación, y ahí es donde está la clase.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.3f}'.format)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('pandas', pd.__version__, '| numpy', np.__version__, '| scipy', __import__('scipy').__version__)

### 1.2 La carga y la limpieza, explicadas

La celda de abajo hace cinco cosas, todas de la clase 3. Léala antes de ejecutarla:

0. **Estandarizar `departamento`.** Espacios, mayúsculas y tildes, la receta de la clase 3. Aquí no
   cambia ningún resultado, porque la llave de esta clase es `c_digo_departamento` y un código no
   tiene ortografía; se hace igual para que la columna sirva si usted la usa en una tabla o en un
   gráfico. **Cuando exista un código, la llave es el código, no el nombre.**
1. **`astype(int)` sobre `ano`.** El archivo mezcla `2018.0` y `2018`. Sin esto, filtrar por periodo
   funciona a veces.
2. **`poblacion_5_16` de texto a número.** Se quita la coma de miles y se convierte con
   `pd.to_numeric(..., errors='coerce')`, que pone `NaN` donde no pueda convertir en vez de reventar.
3. **`drop_duplicates` por (año, departamento).** Hay 20 pares repetidos. Si no se quitan, el `n`
   queda inflado **y todos los intervalos de confianza salen más angostos de lo que corresponde**.
   Ese es el daño concreto que hace un duplicado en esta clase: no un error, una falsa precisión.
4. **La columna `grupo_territorial`.**

**Advertencia metodológica sobre el punto 4, y es importante.** El dataset **no tiene** ninguna columna
de urbano o rural. La construimos nosotros a partir del código DANE del departamento. Eso la convierte
en una **decisión metodológica**, no en un dato, y hay que declararla cada vez que se reporte un
resultado que la use. Si alguien cuestiona el criterio, tiene razón en cuestionarlo: eso es exactamente
lo que un analista debe hacer con la agrupación de otro.

- `Urbanizado`: departamentos con área metropolitana grande.
- `Rural disperso`: Amazonía, Orinoquía y Chocó.
- El resto queda **fuera** de la comparación, y también hay que decirlo.

In [ ]:
# Este cuaderno vive en clase13/demo/, y el CSV dos carpetas más arriba, en datasets/
df_crudo = pd.read_csv('../datos/educacion_estadisticas.csv')
print('Archivo crudo:', df_crudo.shape)

df = df_crudo.copy()

# 0. Nombre del departamento: espacios, mayúsculas y tildes, la receta de la clase 3.
#    Sin esto, 'Guainia' y 'Guainía' son dos departamentos distintos para pandas.
df['departamento'] = (df['departamento'].astype(str)
                      .str.strip()
                      .str.normalize('NFKD')
                      .str.encode('ascii', 'ignore')
                      .str.decode('utf-8')
                      .str.upper())

# 1. Año a entero
df['ano'] = df['ano'].astype(int)

# 2. Población de texto a número
df['poblacion_5_16'] = pd.to_numeric(
    df['poblacion_5_16'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

# 3. Quitar los pares (año, departamento) repetidos
df = df.drop_duplicates(subset=['ano', 'c_digo_departamento']).reset_index(drop=True)

# 4. Grupo territorial: proxy construido por nosotros, NO es una columna del dataset
CODIGOS_URBANIZADO = [11, 5, 76, 8, 68, 25, 66, 17, 63, 13, 54]
CODIGOS_RURAL_DISPERSO = [81, 85, 86, 91, 94, 95, 97, 99, 27]

df['grupo_territorial'] = np.where(
    df['c_digo_departamento'].isin(CODIGOS_URBANIZADO), 'Urbanizado',
    np.where(df['c_digo_departamento'].isin(CODIGOS_RURAL_DISPERSO), 'Rural disperso', 'Otro')
)

print('Después de limpiar:', df.shape)
print()
print(df['grupo_territorial'].value_counts().to_string())

### 1.3 La verificación, que es el punto de toda esta sección

**El concepto: hay errores que no lanzan ninguna excepción.** Si `drop_duplicates` no hubiera corrido,
el cuaderno seguiría funcionando, los intervalos saldrían más angostos y todas las conclusiones serían
más seguras de lo que los datos permiten. Nada en rojo, nada en la consola.

Contra eso solo hay una defensa: **verificar contra un número que usted conozca de antemano.**
Después de deduplicar tienen que quedar 462 filas.

`assert condicion, 'mensaje'` detiene el cuaderno si la condición es falsa. Es la manera de que un
error silencioso deje de serlo.

In [ ]:
assert len(df) == 462, f'Se esperaban 462 filas después de deduplicar, hay {len(df)}'
assert df['ano'].dtype.kind == 'i', 'El año no quedó como entero'

print('Verificación superada.')
print('Filas:', len(df), '| años:', df['ano'].min(), 'a', df['ano'].max())

### 1.4 Mirar la tabla antes de inferir nada

Antes de sacar una sola conclusión sobre lo que **no** se midió, hay que saber qué se midió. Son las
funciones de la clase 2, y aquí no son un repaso: cada una responde una pregunta que esta clase
necesita contestada antes de empezar.

| Función | La pregunta que responde | Por qué importa hoy |
|---------|--------------------------|---------------------|
| `df.shape` | ¿Cuántas filas y columnas quedaron? | El `n` entra directo en el error estándar: `desv / sqrt(n)` |
| `df.head()` / `df.tail()` | ¿Cómo se ven las primeras y las últimas filas? | El final del archivo es donde suelen esconderse los totales y las filas basura |
| `df.info()` | ¿Qué columnas hay, de qué tipo, y cuántas no nulas? | Una columna de texto donde se espera número rompe cualquier prueba |
| `df.dtypes` | ¿De qué tipo quedó cada columna después de limpiar? | Es la comprobación de que `astype(int)` y `to_numeric` hicieron lo suyo |
| `df.describe()` | ¿Qué rango, qué media y qué percentiles tiene cada variable numérica? | Una tasa de matriculación de 300 % o una deserción negativa se ve aquí, no en el t-test |
| `df.isna().sum()` | ¿Cuántos faltantes tiene cada columna? | `scipy` no ignora los `NaN`: si entran, el resultado sale `nan` y no avisa por qué |
| `df.nunique()` | ¿Cuántos valores distintos hay por columna? | 33 departamentos y 14 años: si sale otra cosa, la deduplicación no hizo lo que se creía |
| `value_counts()` | ¿Cómo se reparten las categorías? | Los tamaños de los dos grupos que se van a comparar en la sección 8 |

In [ ]:
print('Forma:', df.shape)
print()
print('Primeras 3 filas de las columnas que usa esta clase:')
print(df[['ano', 'departamento', 'tasa_matriculacion_5_16', 'desercion',
          'grupo_territorial']].head(3).to_string(index=False))
print()
print('Últimas 3 filas de las mismas columnas:')
print(df[['ano', 'departamento', 'tasa_matriculacion_5_16', 'desercion',
          'grupo_territorial']].tail(3).to_string(index=False))

In [ ]:
# info() imprime la radiografía completa: tipo y conteo de no nulos, columna por columna.
df[['ano', 'departamento', 'tasa_matriculacion_5_16', 'desercion',
    'grupo_territorial', 'poblacion_5_16']].info()

In [ ]:
print('Tipos de las columnas que vamos a usar:')
print(df[['ano', 'tasa_matriculacion_5_16', 'desercion', 'poblacion_5_16']].dtypes.to_string())
print()
print('Resumen numérico de las dos variables de la clase:')
print(df[['tasa_matriculacion_5_16', 'desercion']].describe().round(2).to_string())

In [ ]:
print('Faltantes por columna (solo las que usamos):')
print(df[['tasa_matriculacion_5_16', 'desercion', 'poblacion_5_16']].isna().sum().to_string())
print()
print('Valores distintos:')
print(df[['ano', 'departamento', 'c_digo_departamento', 'grupo_territorial']].nunique().to_string())
print()
print('Reparto de los grupos que se comparan en la sección 8:')
print(df['grupo_territorial'].value_counts().to_string())

**Pregunta de interpretación 1.** Mire tres salidas juntas: los faltantes de `desercion`, el
`describe()` de las dos variables y el reparto de `grupo_territorial`. Diga qué le pasaría a la
prueba de la sección 8 si esos faltantes entraran sin quitarse, y por qué el tamaño del grupo
`Rural disperso` importa antes de leer cualquier p-valor.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

`desercion` tiene faltantes, y `scipy` no los ignora: si entran a `ttest_ind`, el estadístico y el
p-valor salen `nan`. No hay excepción, no hay mensaje en rojo, sale un `nan` que alguien puede
copiar a un informe. Por eso cada prueba de este cuaderno llama a `.dropna()` antes, y por eso la
sección 5 lo muestra como el primero de los tres errores que no avisan.

El tamaño de `Rural disperso` importa por dos razones distintas. La primera es aritmética: el error
estándar es `desv / sqrt(n)`, así que el grupo más pequeño es el que manda el ancho del intervalo de
la diferencia. La segunda es de interpretación, y es la que se olvida: con `n` chico un efecto real
puede salir no significativo, y "no significativo" no quiere decir "no hay diferencia", quiere decir
que estos datos no alcanzan para distinguirla del azar.

Y el `describe()` es el filtro previo a todo lo demás: si la matriculación tuviera un 300 % o la
deserción un valor negativo, ninguna prueba de hipótesis lo detectaría. Reportaría con toda
seriedad una diferencia entre dos grupos calculada sobre un imposible.

</details>

---

## 2. Muestra y población

### Para entender qué está pasando · La sopa

Usted está cocinando una olla de sopa para cuarenta personas. Quiere saber si le falta sal. **No se
toma la olla entera**: revuelve, prueba una cucharada y decide.

Esa cucharada es una **muestra**. La olla es la **población**. Y ahí está todo lo que hace la
estadística inferencial:

- Nadie mide la población completa. Nunca. Ni el DANE, ni el Ministerio, ni usted en su proyecto.
- Se mide una parte y se **afirma algo sobre el todo**.
- Esa afirmación puede estar equivocada, y lo importante es poder decir **cuánto** puede estarlo.

Dos condiciones que la analogía hace evidentes y que se olvidan siempre:

1. **Hay que revolver.** Si prueba la cucharada de la superficie, mide la superficie, no la sopa. Una
   muestra que no representa a la población no se arregla con más matemáticas. Este es el problema del
   **sesgo de muestreo**, y es el único de la clase que ninguna fórmula resuelve.
2. **El tamaño de la cucharada importa, el tamaño de la olla casi no.** Para una olla de cuarenta
   personas y una de cuatrocientas se prueba una cucharada igual. Suena falso y es cierto: lo que manda
   es el tamaño de la muestra, no qué fracción de la población representa.

**Y qué es la población en nuestro caso.** Las 462 filas del archivo son lo que el Ministerio midió.
La población —la realidad completa del sistema educativo colombiano en esos años, con todas sus
variaciones— no está en ningún archivo y nunca la vamos a ver.

### 2.2 La trampa deliberada: fabricar una población para poder verla

Para **ver** el mecanismo hay que hacer algo que en la vida real es imposible: conocer la población.
Así que hacemos trampa a propósito y **declaramos** que estas 462 filas son la población completa.

No es cierto, y por eso se dice en voz alta. Pero es la única forma de responder la pregunta que
importa: *si yo solo hubiera medido 30 departamentos-año en vez de 462, ¿qué tan lejos habría quedado
mi respuesta de la verdad?*

**La función de andamiaje.** `muestras_repetidas(...)` sortea muchas muestras de un tamaño dado. Ya
está escrita, y tiene un detalle que no es decoración: recibe una **semilla**
(`np.random.default_rng(semilla)`). La semilla fija el azar, de modo que a todos en el salón les salga
exactamente lo mismo. Sin semilla, cada ejecución daría números distintos y no habría nada que comparar
ni nada que verificar.

`replace=True` significa **muestreo con reemplazo**: cada dato puede salir más de una vez. Es la forma
de simular "otro estudio que mide desde la misma realidad", que es justo lo que queremos imitar.

In [ ]:
SEMILLA_MUESTREO = 13

POBLACION = df['tasa_matriculacion_5_16'].dropna().to_numpy()
MEDIA_POBLACIONAL = POBLACION.mean()

print(f'Tamaño de nuestra "población": {len(POBLACION)}')
print(f'Media poblacional (el número que en la vida real NUNCA se ve): {MEDIA_POBLACIONAL:.4f} %')


def muestras_repetidas(valores, tamano, repeticiones, semilla):
    """Sortea 'repeticiones' muestras de tamaño 'tamano', con reemplazo y semilla fija."""
    generador = np.random.default_rng(semilla)
    return [generador.choice(valores, size=tamano, replace=True) for _ in range(repeticiones)]


print('Función de muestreo definida.')

### 2.3 Tres estudios distintos sobre la misma realidad

La celda de abajo toma **tres** muestras de 30 datos cada una. Las tres salen de la misma población,
las tres están bien tomadas, ninguna tiene error.

Mire las tres medias.

In [ ]:
tres = muestras_repetidas(POBLACION, tamano=30, repeticiones=3, semilla=SEMILLA_MUESTREO)

for i, muestra in enumerate(tres, start=1):
    print(f'Estudio {i}: n = {len(muestra)}, media muestral = {muestra.mean():.2f} %')

print()
print(f'La verdad (que ninguno de los tres puede ver): {MEDIA_POBLACIONAL:.2f} %')
print()
print('Ninguno de los tres se equivocó. Los tres midieron bien. Y les dio distinto.')
print('Eso es la variabilidad de muestreo, y no es un error: es la condición del oficio.')

### 2.4 Mil estudios, y la forma que aparece

Si tres medias ya son distintas, ¿cómo se ven mil?

**Distribución muestral** es el nombre de lo que va a salir: la distribución de un estadístico —aquí,
la media— a lo largo de muchas muestras. No es la distribución de los datos: es la distribución de las
**respuestas** de muchos estudios.

Y aparece un hecho que sostiene todo lo que viene después: **la distribución de las medias muestrales
tiende a ser normal, aunque los datos originales no lo sean.** Es el **teorema del límite central**.
No lo vamos a demostrar; lo vamos a ver en el gráfico de la derecha.

In [ ]:
muestras_mil = muestras_repetidas(POBLACION, tamano=30, repeticiones=1000,
                                  semilla=SEMILLA_MUESTREO)
medias_mil = np.array([m.mean() for m in muestras_mil])

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

ejes[0].hist(POBLACION, bins=30, color='#7f8c8d')
ejes[0].axvline(MEDIA_POBLACIONAL, color='black', lw=2)
ejes[0].set_title('Los DATOS: 462 tasas de matriculación')
ejes[0].set_xlabel('Tasa de matriculación (%)')
ejes[0].set_ylabel('Frecuencia')

ejes[1].hist(medias_mil, bins=30, color='#2980b9')
ejes[1].axvline(MEDIA_POBLACIONAL, color='black', lw=2)
ejes[1].set_title('Las MEDIAS de 1.000 estudios de n = 30')
ejes[1].set_xlabel('Media muestral (%)')
ejes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print(f'Los datos van de {POBLACION.min():.1f} a {POBLACION.max():.1f}')
print(f'Las medias van de {medias_mil.min():.1f} a {medias_mil.max():.1f}')
print()
print('Dos cosas para llevarse:')
print(' 1. Las medias se apiñan mucho más que los datos. Promediar cancela ruido.')
print(' 2. La forma de la derecha es acampanada aunque la de la izquierda no lo sea.')

**Pregunta de interpretación 2.** Los dos histogramas están hechos con los mismos datos y salen
distintos: el de la izquierda es ancho y feo, el de la derecha es angosto y acampanado. ¿Qué hay
dibujado en cada uno? Y si mañana alguien le dice *"para hacer inferencia sus datos tienen que ser
normales"*, ¿qué le responde con este gráfico a la vista?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No están dibujadas las mismas cosas.** A la izquierda están los **datos**: 462 tasas de
matriculación, cada punto un departamento-año. A la derecha están las **respuestas de 1.000 estudios**:
cada punto es la media de una muestra de 30. Son dos poblaciones de números distintas, y confundirlas
es el origen de casi todos los errores de esta clase.

**Por qué la de la derecha es más angosta:** promediar cancela ruido. Un valor extremo que mueve
muchísimo el histograma de la izquierda queda diluido entre otros 29 cuando se promedia. Esa es la
razón de fondo por la que se puede estimar bien con muestras chicas.

**Y la respuesta a la frase:** lo que tiene que tender a normal es la **distribución de la media**, no
la de los datos. Eso es el teorema del límite central, y es exactamente lo que muestra el panel
derecho: los datos de la izquierda no son normales y las medias de la derecha sí se acercan. Por eso el
t-test aguanta datos torcidos cuando el n es decente. Lo que no aguanta es un n minúsculo con una
distribución muy asimétrica, y ahí es donde entran las pruebas no paramétricas.

</details>

### 2.5 El error estándar: el ancho de esa segunda campana

**Qué es el error estándar.** La desviación estándar **de la distribución muestral**. Dicho en
español: qué tanto se movería su respuesta si repitiera el estudio.

Y aquí está la distinción que más confusiones evita en toda la clase:

| | Qué mide | Se achica con más datos |
|--|----------|--------------------------|
| **Desviación estándar** | Qué tan dispersos están los **datos** alrededor de su media | No. Es una propiedad de la realidad |
| **Error estándar** | Qué tan dispersa estaría **la media** si repitiera el estudio | **Sí.** Es una propiedad de su estudio |

La fórmula es corta:

```
error estándar = desviación estándar / raíz de n
```

Fíjese en la raíz: **cuadruplicar la muestra reduce el error estándar a la mitad, no a la cuarta
parte.** Los datos se compran caros y rinden a la raíz. Es la razón por la que "toma más datos" es
buen consejo hasta cierto punto y después deja de serlo.

La celda de abajo comprueba la fórmula contra la simulación: la desviación estándar de las 1.000 medias
que acabamos de calcular tiene que parecerse a la fórmula aplicada sobre una sola muestra.

In [ ]:
desviacion_poblacional = POBLACION.std(ddof=1)
error_estandar_teorico = desviacion_poblacional / np.sqrt(30)
error_estandar_simulado = medias_mil.std(ddof=1)

print(f'Desviación estándar de los datos          : {desviacion_poblacional:.4f}')
print(f'Error estándar por la fórmula (n = 30)    : {error_estandar_teorico:.4f}')
print(f'Dispersión REAL de las 1.000 medias       : {error_estandar_simulado:.4f}')
print()
print('La fórmula no es una convención: predice lo que la simulación produce.')
print('Y note el tamaño: el error estándar es una fracción de la desviación estándar.')

### 2.6 El error estándar de nuestro estudio

Volvemos al estudio real: las 462 filas del archivo, no una muestra de 30.

La celda de abajo saca los tres números que hacen falta —`n`, la media y la desviación estándar
muestral— y la siguiente aplica la fórmula a mano y la compara con `stats.sem`, que hace exactamente lo
mismo. **Antes de ejecutarlas, prediga:** ¿el error estándar de este estudio va a ser más grande o más
chico que el de n = 30 que acaba de ver?

In [ ]:
matriculacion = df['tasa_matriculacion_5_16'].dropna()

n = len(matriculacion)
media = matriculacion.mean()
desv = matriculacion.std(ddof=1)     # ddof=1 = desviación estándar muestral

print(f'n          = {n}')
print(f'media      = {media:.4f} %')
print(f'desv. est. = {desv:.4f} %')

In [ ]:
error_estandar_manual = desv / np.sqrt(n)

print(f'Error estándar a mano  = {error_estandar_manual:.4f}')
print(f'Error estándar (scipy) = {stats.sem(matriculacion):.4f}')
print(f'Coinciden: {np.isclose(error_estandar_manual, stats.sem(matriculacion))}')
print()
print(f'Para comparar, el de un estudio de n = 30: {desv / np.sqrt(30):.4f}')

`stats.sem(x)` hace exactamente eso mismo. No hay magia dentro de scipy: hay la fórmula de arriba,
desviación estándar entre raíz de n.

**Pregunta de interpretación 3.** El error estándar de las 462 filas es mucho más chico que el de una
muestra de 30, y la desviación estándar de los datos no se movió. Explique las dos cosas con la fórmula
en la mano. Y responda la pregunta que le van a hacer en la sustentación: si con 462 datos su intervalo
todavía le parece ancho, ¿cuántos datos necesitaría para dejarlo en la mitad de ancho?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**La desviación estándar no se movió porque no es suya: es de la realidad.** Mide qué tan dispersas
están las tasas de matriculación entre departamentos y años. Medir más departamentos no vuelve a
Colombia más homogénea.

**El error estándar sí se movió porque es una propiedad de su estudio.** Mide cuánto se movería su
respuesta si repitiera la medición, y eso depende de cuántos datos usó: `desv / raíz(n)`. Con n = 462
en vez de 30, el denominador es casi cuatro veces más grande.

**Y la respuesta incómoda: para partir el ancho a la mitad hacen falta cuatro veces más datos.** La
raíz es la que manda. Pasar de 462 a 924 no lo mejora al doble, lo mejora en un 30% aproximado; para el
doble de precisión hay que llegar a unos 1.848. Por eso "consigamos más datos" es buen consejo al
principio y deja de serlo rápido: los datos se compran caros y rinden a la raíz. Cuando un intervalo es
demasiado ancho, muchas veces la salida no es más muestra sino medir mejor, o partir la pregunta en una
más estrecha.

</details>

---

## 3. El intervalo de confianza

### La analogía de la arquería

El arquero dispara a una diana. El **centro exacto** de la diana es el valor real de la población: la
tasa de matriculación verdadera de Colombia. Nunca la vemos. Solo vemos dónde cayó la flecha.

- **Estimación puntual** = una flecha. *"La tasa es 85,53%"*. Precisa, y casi con certeza equivocada.
- **Intervalo de confianza** = declarar una zona de la diana **antes** de disparar. *"La tasa está entre
  84,58% y 86,48%"*. Menos precisa, correcta muchas más veces.

**El trade-off, dicho sin adornos:** intervalo angosto es una afirmación fuerte con más riesgo de
errar; intervalo ancho es una afirmación débil con menos riesgo. Un intervalo de 0% a 100% nunca se
equivoca y no sirve para nada. El 95% es donde la convención decidió pararse.

**Qué mueve el ancho del intervalo** (esto sí conviene memorizarlo):

| Palanca | Efecto en el ancho | ¿La controla usted? |
|---------|--------------------|---------------------|
| Más datos (n más grande) | Más angosto | **Sí.** Es la única que realmente controla |
| Más dispersión en los datos | Más ancho | No: es la realidad |
| Más confianza pedida (99% en vez de 95%) | Más ancho | Sí, y se paga en precisión |

### La fórmula

```
IC = media  +/-  t * (desviación estándar / raíz de n)
                      \_________________________/
                            error estándar
```

`t` es un multiplicador que sale de la distribución t de Student y depende del nivel de confianza y del
tamaño de la muestra. No hay que calcularlo a mano: `stats.t.interval` lo hace.

### `stats.t.interval`, argumento por argumento

| Argumento | Qué es | Qué le pasamos |
|-----------|--------|----------------|
| primero (posicional) | Nivel de confianza | `0.95` |
| `df` | Grados de libertad | `n - 1` |
| `loc` | Centro del intervalo | la media muestral |
| `scale` | Escala | el **error estándar**, no la desviación estándar |

**El error más caro de esta función es pasarle la desviación estándar en `scale`.** No lanza ningún
error: devuelve un intervalo absurdamente ancho que parece un resultado. Lo provocamos a propósito en
la sección 5.

Ojo con el nombre del argumento `df`: aquí significa *degrees of freedom*, grados de libertad. No tiene
nada que ver con el DataFrame que también se llama `df`. Es una colisión de nombres desafortunada de la
librería, y confunde a todo el mundo la primera vez.

In [ ]:
ic_95 = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(matriculacion))

print(f'Media muestral : {media:.2f} %')
print(f'IC 95%         : ({ic_95[0]:.2f} , {ic_95[1]:.2f})')
print(f'Ancho          : {ic_95[1] - ic_95[0]:.2f} puntos porcentuales')

### 3.2 El mismo cálculo al 99%

**Antes de ejecutar la celda, decida:** ¿el intervalo al 99% va a ser más ancho o más angosto que el
del 95%? Escríbalo aquí y después ejecute.

*Mi predicción:*

In [ ]:
ic_99 = stats.t.interval(0.99, df=n - 1, loc=media, scale=stats.sem(matriculacion))

print(f'IC 95% : ({ic_95[0]:.2f} , {ic_95[1]:.2f})   ancho = {ic_95[1] - ic_95[0]:.2f}')
print(f'IC 99% : ({ic_99[0]:.2f} , {ic_99[1]:.2f})   ancho = {ic_99[1] - ic_99[0]:.2f}')
print()
print('Más confianza se paga con menos precisión. Pedir acertar 99 de cada 100 veces')
print('en vez de 95 obliga a apuntar a una zona más grande de la diana.')

### 3.3 Los tres anchos, uno al lado del otro

La misma llamada, cambiando **un solo argumento**: la confianza. Al 90%, al 95% y al 99%, sobre la
misma variable y la misma muestra.

In [ ]:
ic_90 = stats.t.interval(0.90, df=n - 1, loc=media, scale=stats.sem(matriculacion))
ancho_ic_90 = ic_90[1] - ic_90[0]

print(f'IC 90% : ({ic_90[0]:.2f} , {ic_90[1]:.2f})   ancho = {ancho_ic_90:.4f}')
print(f'IC 95% : ({ic_95[0]:.2f} , {ic_95[1]:.2f})   ancho = {ic_95[1] - ic_95[0]:.4f}')
print(f'IC 99% : ({ic_99[0]:.2f} , {ic_99[1]:.2f})   ancho = {ic_99[1] - ic_99[0]:.4f}')

**Pregunta de interpretación 4.** Un intervalo angosto suena mejor que uno ancho: parece más preciso.
Con los tres anchos a la vista, explique qué se está pagando al pasar del 99% al 90%. Y conteste
derecho: ¿puede un analista elegir la confianza **después** de ver los tres resultados, quedándose con
la que le conviene?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Lo que se paga es tasa de acierto.** El intervalo al 90% es más angosto porque afirma más, y a cambio
falla más seguido: se equivoca en 1 de cada 10 estudios en vez de 1 de cada 20 o 1 de cada 100. Nada
mejoró al angostarlo; solo se movió el reparto entre precisión y prudencia. Angosto no es "mejor", es
"más arriesgado".

**Y no, no se puede elegir después.** Escoger el nivel de confianza mirando cuál deja el resultado más
vistoso es la misma trampa que escribir H0 después de ver el p-valor: convierte una decisión de método
en una decisión de conveniencia. El nivel se fija **antes**, se declara en el informe y se sostiene aunque
el resultado no guste. Lo que sí es legítimo es justificarlo por el costo de equivocarse: 99% cuando el
error es caro (salud, seguridad, plata), 90% en un tanteo exploratorio que después se va a confirmar.

</details>

---

## 4. Qué significa el 95%, y qué no

### Para entender qué está pasando · La parte que casi todo el mundo dice mal

Esta es **la** sección de la clase. Si de hoy se lleva una sola cosa, que sea esta.

**Incorrecto:** *"hay un 95% de probabilidad de que el valor real esté en este intervalo".*

**Correcto:** *"si repitiera este estudio muchas veces, alrededor del 95% de los intervalos que
construya contendrían el valor real".*

**Por qué la primera está mal.** El valor real es **fijo**: la tasa verdadera de Colombia es un número,
no una variable aleatoria. Lo aleatorio es el **intervalo**, porque depende de qué muestra le tocó.
Este intervalo suyo, el que ya calculó, o contiene el valor real o no lo contiene. No hay probabilidad
de por medio: hay ignorancia.

**El 95% es una propiedad del procedimiento, no de este intervalo.** Es la tasa de acierto del método a
lo largo de muchos estudios. Comparable con: un cirujano con 95% de éxito no le da a usted un 95%
después de operarlo; su operación ya salió bien o ya salió mal.

Esto suena a sutileza de profesor y no lo es. Es la diferencia entre reportar *"estamos casi seguros de
que la deserción es 4%"* y *"nuestro método acierta 19 de cada 20 veces, y esta vez dijo entre 3,9% y
4,2%"*. La segunda es defendible; la primera, no.

### Y ahora, en vez de creerlo, lo vemos

Nosotros sí conocemos la media poblacional: la fabricamos en la sección 2. Entonces podemos hacer lo
que en la vida real es imposible: **construir 200 intervalos a partir de 200 estudios distintos y
contar en cuántos cae la verdad.**

Si la teoría sirve, tienen que acertar alrededor de 190 de los 200.

In [ ]:
REPETICIONES_IC = 200
TAMANO_ESTUDIO = 30

muestras_ic = muestras_repetidas(POBLACION, tamano=TAMANO_ESTUDIO,
                                 repeticiones=REPETICIONES_IC, semilla=7)


def intervalo(muestra, confianza):
    """El intervalo de confianza de la media de una muestra. Devuelve (inferior, superior)."""
    return stats.t.interval(confianza, df=len(muestra) - 1,
                            loc=muestra.mean(), scale=stats.sem(muestra))


aciertos = 0
for muestra in muestras_ic:
    inferior, superior = intervalo(muestra, 0.95)
    if inferior <= MEDIA_POBLACIONAL <= superior:
        aciertos += 1

print(f'Estudios simulados            : {REPETICIONES_IC}')
print(f'Intervalos que atraparon la verdad: {aciertos}')
print(f'Tasa de acierto               : {aciertos / REPETICIONES_IC:.1%}')
print()
print('Eso, y solo eso, es el "95% de confianza": la tasa de acierto del método.')

### 4.2 Los mismos intervalos, dibujados

Cada línea horizontal es un estudio. La línea vertical negra es la verdad, que ningún estudio ve. Los
intervalos rojos son los que fallaron.

Este gráfico es el dibujo de los cinco estudios que el profesor deja en el tablero, con 40 estudios en
vez de 5.

In [ ]:
A_DIBUJAR = 40

fig, eje = plt.subplots(figsize=(9, 8))

for i, muestra in enumerate(muestras_ic[:A_DIBUJAR]):
    inferior, superior = intervalo(muestra, 0.95)
    acerto = inferior <= MEDIA_POBLACIONAL <= superior
    eje.plot([inferior, superior], [i, i],
             color='#2980b9' if acerto else '#c0392b',
             lw=2.2 if acerto else 3)
    eje.plot(muestra.mean(), i, 'o', ms=3,
             color='#2980b9' if acerto else '#c0392b')

eje.axvline(MEDIA_POBLACIONAL, color='black', lw=2)
eje.text(MEDIA_POBLACIONAL, A_DIBUJAR + 0.5, ' la verdad', fontsize=10, va='bottom')
eje.set_xlabel('Tasa de matriculación (%)')
eje.set_ylabel('Estudio simulado')
eje.set_title(f'{A_DIBUJAR} estudios, {A_DIBUJAR} intervalos al 95%\n'
              'En rojo, los que no contienen el valor real')
plt.tight_layout()
plt.show()

fallidos = sum(1 for m in muestras_ic[:A_DIBUJAR]
               if not intervalo(m, 0.95)[0] <= MEDIA_POBLACIONAL <= intervalo(m, 0.95)[1])
print(f'De los {A_DIBUJAR} dibujados, fallaron {fallidos}.')
print('Los rojos no están mal calculados. Son la parte del 5% que el método promete fallar.')

**Pregunta de interpretación 5.** Mire un intervalo rojo concreto del gráfico. Ese estudio hizo todo
bien: tomó su muestra, aplicó la fórmula correcta y publicó su intervalo. ¿En qué se equivocó? Y si
usted fuera el analista de ese estudio, ¿habría podido saber que le tocó ser uno de los rojos?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No se equivocó en nada.** Ese es exactamente el punto y por eso duele. El procedimiento fue correcto
de principio a fin; le tocó una muestra que, por azar, cayó lejos del centro. El método promete fallar
alrededor de 5 de cada 100 veces, y ese fue uno de esos.

**Y no, no habría podido saberlo.** Desde dentro de un solo estudio no hay forma de distinguir un
intervalo azul de uno rojo: se ven idénticos. La única manera de verlo es conocer la verdad, y si uno
conociera la verdad no estaría estimando nada.

De ahí sale la consecuencia práctica: **la confianza no se pone en un intervalo, se pone en el
procedimiento.** Un resultado aislado nunca es concluyente, y por eso en ciencia se replica. Cuando
alguien le muestre un solo estudio con un solo intervalo y le diga "está probado", usted ya sabe qué
preguntar.

</details>

### 4.3 La misma cuenta, al 80%

Si el porcentaje de confianza es de verdad la tasa de acierto del método, entonces pedir 80% tiene que
producir alrededor de 0,80 de aciertos, no 0,95.

La celda de abajo repite el conteo sobre las **mismas** 200 muestras —`muestras_ic`, ya sorteadas con
semilla fija— cambiando solo la confianza. Usar las mismas muestras es lo que hace comparables los dos
números: si se volvieran a sortear, la diferencia mezclaría dos cosas.

In [ ]:
aciertos_80 = 0
for muestra in muestras_ic:
    inferior, superior = intervalo(muestra, 0.80)
    if inferior <= MEDIA_POBLACIONAL <= superior:
        aciertos_80 += 1

cobertura_80 = aciertos_80 / len(muestras_ic)

print(f'Cobertura al 80% : {cobertura_80:.1%}   ({aciertos_80} de {REPETICIONES_IC})')
print(f'Cobertura al 95% : {aciertos / REPETICIONES_IC:.1%}   ({aciertos} de {REPETICIONES_IC})')
print()
print('El porcentaje de confianza no es una etiqueta: es la tasa de acierto que se cumple.')

**Pregunta de interpretación 6.** Las dos coberturas salieron cerca de lo prometido, pero no clavadas
en 80,0% y 95,0%. Primero: ¿por qué no dan exactas, y qué habría que cambiar para acercarlas más?
Segundo: el intervalo al 80% es más angosto, o sea que afirma más, y falla más. Dé un caso concreto de
su vida profesional donde preferiría el 80%, y uno donde exigiría el 99%.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**No dan exactas porque 200 estudios también son una muestra.** La cobertura observada es en sí misma
una estimación, con su propia variabilidad. Con 2.000 o 20.000 repeticiones se pegaría más al valor
prometido. Es el mismo fenómeno de la sección 2 aplicado un nivel más arriba, y conviene notarlo:
incluso el número que verifica el método está sujeto a azar de muestreo.

**Cuándo el 80%:** un tanteo interno y barato de revertir. Un piloto para decidir si vale la pena
seguir explorando una hipótesis, un tablero de seguimiento semanal donde equivocarse cuesta una reunión.
Ahí la precisión ayuda a decidir rápido y el error se corrige en la siguiente medición.

**Cuándo el 99%:** cuando el error se paga caro y no se devuelve. Un lote de medicamentos, un umbral de
seguridad estructural, una cifra que va a un informe público o a un contrato. Ahí un intervalo ancho es
información honesta, no debilidad.

**La regla de fondo:** el nivel de confianza es una decisión de negocio disfrazada de decisión
estadística. Lo fija el costo de equivocarse, no la costumbre.

</details>

---

## 5. Los tres errores que no avisan

Todos los de esta sección tienen la misma firma: **el código corre, no hay nada en rojo, y el resultado
está mal.** Son los que hay que provocar una vez para reconocerlos después.

### 5.1 El `dropna()` que falta

`NaN` es contagioso: cualquier operación aritmética que lo toque devuelve `NaN`. La media de una
columna con un solo faltante ya es `NaN`, y `NaN` no lanza ninguna excepción.

La columna `desercion` tiene 47 faltantes en las 462 filas. Mire qué pasa si no se limpian.

In [ ]:
desercion_sucia = df['desercion']              # con los NaN adentro
desercion_limpia = df['desercion'].dropna()   # sin ellos

print(f'Filas con dato : {desercion_limpia.shape[0]} de {desercion_sucia.shape[0]}')
print()
print(f'Media SIN dropna : {desercion_sucia.mean():.4f}   <- pandas ignora los NaN al promediar')
print(f'Media CON dropna : {desercion_limpia.mean():.4f}')
print()
ic_sucio = stats.t.interval(0.95, df=len(desercion_sucia) - 1,
                            loc=desercion_sucia.mean(), scale=stats.sem(desercion_sucia))
print(f'IC 95% sin dropna : {ic_sucio}   <- aquí sí se rompió, y en silencio')
print()
print('pandas ignora los NaN al promediar; scipy NO los ignora al calcular el error estándar.')
print('Resultado: una media razonable y un intervalo que es (nan, nan). Ningún error, ningún aviso.')
print('Regla: dropna() ANTES de calcular. Siempre. Y se reporta el n que quedó.')

### 5.2 La desviación estándar en `scale`

El segundo, y es el más caro de esta clase. `stats.t.interval` acepta cualquier número en `scale`. Si
le pasa la desviación estándar en vez del error estándar, devuelve un intervalo perfectamente formado y
completamente equivocado.

La pista para detectarlo a ojo: **el intervalo sale del orden de la dispersión de los datos, no de la
precisión de la media.** Si su intervalo para un porcentaje va de 65% a 106%, algo pasó.

In [ ]:
ic_bien = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(matriculacion))
ic_mal = stats.t.interval(0.95, df=n - 1, loc=media, scale=desv)   # <- desviación, no error estándar

print(f'Con el error estándar (correcto)  : ({ic_bien[0]:.2f} , {ic_bien[1]:.2f})')
print(f'Con la desviación estándar (mal)  : ({ic_mal[0]:.2f} , {ic_mal[1]:.2f})')
print()
print(f'El segundo es {(ic_mal[1] - ic_mal[0]) / (ic_bien[1] - ic_bien[0]):.0f} veces más ancho.')
print('Y es exactamente la raíz de n, que es lo que se olvidó dividir.')
print()
print('Ninguna de las dos llamadas dio error. Las dos devolvieron una tupla de dos números.')

### 5.3 La prueba equivocada: `ttest_rel` en vez de `ttest_ind`

El tercero sí avisa, y por eso conviene verlo ahora: es el único de la clase que se anuncia con una
excepción, y hay que saber leerla.

- **`ttest_ind`** compara **dos grupos independientes**: personas distintas, departamentos distintos.
- **`ttest_rel`** compara **datos apareados**: los mismos sujetos medidos dos veces (antes y después).

Nuestros dos grupos territoriales son independientes y tienen tamaños distintos. `ttest_rel` exige que
tengan la misma longitud, porque empareja fila con fila.

`try / except` ejecuta el bloque y, si lanza una excepción, la atrapa en vez de detener el cuaderno.
Lo usamos para provocar el error a propósito y leer el mensaje con calma.

In [ ]:
urbano = df[df['grupo_territorial'] == 'Urbanizado']['desercion'].dropna()
rural = df[df['grupo_territorial'] == 'Rural disperso']['desercion'].dropna()

print(f'n urbanizado    : {len(urbano)}')
print(f'n rural disperso: {len(rural)}')
print()

try:
    stats.ttest_rel(urbano, rural)
except Exception as error:
    print('ttest_rel ->', type(error).__name__)
    print('  ', error)

print()
print('El mensaje habla de longitudes o de formas que no coinciden. Esa es la pista:')
print('ttest_rel empareja fila con fila, y aquí no hay nada que emparejar.')
print('Departamentos urbanizados y rurales son grupos independientes: va ttest_ind.')

**Pregunta de interpretación 7.** Acaba de ver tres errores: el `dropna()` que falta, la desviación
estándar metida en `scale` y la prueba equivocada. Ordénelos de más peligroso a menos peligroso **para
usted, en la sustentación**, y justifique el orden. Pista: no lo ordene por gravedad estadística.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**El más peligroso es el segundo: la desviación estándar en `scale`.** Devuelve un intervalo
perfectamente formado, con dos números plausibles, y no hay nada que lo delate salvo que a usted le
llame la atención el ancho. Se puede llevar a una sustentación entero y sin que nadie lo note hasta que
alguien pregunte por qué su intervalo de un porcentaje se sale de 100.

**El segundo es el `dropna()` que falta**, y está por debajo solo porque suele delatarse: la media sale
razonable pero el intervalo sale `(nan, nan)`, y un `nan` en pantalla se ve. El daño verdadero de este
no es el `nan`, es la versión silenciosa: reportar un `n` que no corresponde a las filas que realmente
entraron al cálculo.

**El menos peligroso es el tercero, la prueba equivocada**, precisamente porque revienta. Una excepción
es un error que se arregla; un resultado bonito y falso es un error que se publica.

**El criterio del orden, entonces, no es estadístico: es cuánto avisa.** Los errores que gritan son
baratos. Los que salen limpios son los que hay que aprender a cazar a mano, y contra esos la única
defensa es la de la sección 1.3: verificar contra un número que usted conozca de antemano.

</details>

---

## 6. La prueba de hipótesis

### La analogía del tribunal

| Tribunal | Prueba de hipótesis |
|----------|---------------------|
| El acusado es inocente hasta que se demuestre lo contrario | **H0**: no hay efecto, no hay diferencia |
| La fiscalía acusa | **H1**: sí hay efecto, sí hay diferencia |
| Las pruebas presentadas | Los datos de la muestra |
| "Más allá de toda duda razonable" | El nivel **alfa** (0,05 por convención) |
| Veredicto: culpable | Rechazamos H0 |
| Veredicto: **no culpable** | **No rechazamos** H0 |

**El punto que hay que clavar:** un tribunal nunca declara "inocente". Declara "no culpable", que no es
lo mismo. "No culpable" significa que la fiscalía no presentó pruebas suficientes, no que el acusado no
lo haya hecho.

Igual aquí: **nunca se acepta H0, solo se falla en rechazarla.** Ausencia de evidencia no es evidencia
de ausencia. Escribir "se demostró que no hay diferencia" es el error que más caro cuesta en un informe,
y en la rúbrica de hoy le pone techo a la dimensión Saber.

**H0 siempre es la aburrida:** no hay diferencia, no hay efecto, todo sigue igual. Se elige así porque
es la que se puede poner a prueba: permite calcular qué tan raros serían los datos si fuera cierta.

### Los 4 pasos, otra vez

1. **Plantear.** H0 y H1, en español, **antes** de mirar los datos.
2. **Elegir alfa.** 0,05 salvo que haya razón para otra cosa.
3. **Calcular.** Descriptivos primero, después el estadístico y el p-valor.
4. **Decidir y redactar.** p < alfa, se rechaza H0. Y se redacta en lenguaje de negocio.

### Para entender qué está pasando · Qué es un p-valor

**La definición, que conviene copiar literal:**

> El p-valor es la probabilidad de observar datos así de extremos, **o más**, **si H0 fuera cierta**.

Las cinco últimas palabras son las que casi nunca se dicen y las que lo cambian todo. El p-valor **parte
de suponer** que no hay efecto, y mide qué tan raros serían sus datos bajo ese supuesto. No dice nada
sobre si el supuesto es cierto.

**Y ahora, en vez de creerlo, lo vemos.** La celda de abajo hace 2.000 pruebas de hipótesis en un mundo
donde **H0 es literalmente cierta por construcción**: los dos grupos que compara salen de la misma
población. No hay ninguna diferencia real, ni una.

Si el p-valor es lo que dice la definición, entonces en ese mundo los p-valores tienen que repartirse
por igual entre 0 y 1, y alrededor del 5% de ellos tienen que caer por debajo de 0,05 **sin que exista
ninguna diferencia**.

Eso último es la definición operativa de alfa: **la tasa de falsos positivos que usted acepta de
antemano.**

In [ ]:
REPETICIONES_H0 = 2000
TAMANO_GRUPO = 40
generador_h0 = np.random.default_rng(2013)

p_valores_h0 = []
for _ in range(REPETICIONES_H0):
    grupo_a = generador_h0.choice(POBLACION, size=TAMANO_GRUPO, replace=True)
    grupo_b = generador_h0.choice(POBLACION, size=TAMANO_GRUPO, replace=True)
    p_valores_h0.append(stats.ttest_ind(grupo_a, grupo_b, equal_var=False).pvalue)

p_valores_h0 = np.array(p_valores_h0)

fig, eje = plt.subplots(figsize=(9, 4.5))
eje.hist(p_valores_h0, bins=20, color='#7f8c8d', edgecolor='white')
eje.axvline(0.05, color='#c0392b', lw=2)
eje.text(0.055, eje.get_ylim()[1] * 0.9, 'alfa = 0,05', color='#c0392b', fontsize=10)
eje.set_xlabel('p-valor obtenido')
eje.set_ylabel('Número de pruebas')
eje.set_title(f'{REPETICIONES_H0} pruebas en un mundo donde H0 es CIERTA')
plt.tight_layout()
plt.show()

print(f'Pruebas con p < 0,05 : {(p_valores_h0 < 0.05).sum()} de {REPETICIONES_H0} '
      f'({(p_valores_h0 < 0.05).mean():.1%})')
print()
print('Ninguna de esas pruebas encontró nada, porque no había nada que encontrar.')
print('Y aun así, alrededor del 5% dijo "significativo". Eso NO es un defecto del método:')
print('es lo que alfa = 0,05 significa. Usted acepta equivocarse 1 de cada 20 veces.')

**Pregunta de interpretación 8.** Mire el histograma: los p-valores se reparten casi parejo entre 0 y 1,
y alrededor de un centenar de las 2.000 pruebas dio "significativo" sin que existiera ninguna
diferencia. Ahora traduzca eso a su proyecto: si su equipo prueba veinte comparaciones distintas sobre
su dataset y reporta solo la que dio p < 0,05, ¿qué acaba de fabricar? ¿Y qué habría que reportar para
que el hallazgo siga siendo defendible?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Fabricó un hallazgo que no existe, y el gráfico dice con qué probabilidad.** Con veinte pruebas
independientes sobre datos sin ninguna señal, la probabilidad de que al menos una salga significativa al
0,05 es de alrededor del 64%. O sea: es más probable que salga un "hallazgo" a que no salga ninguno.
Reportar solo esa es **p-hacking**, y no hace falta mala fe para caer en él: basta con seguir probando
hasta que algo funcione.

**Lo que hay que reportar:** todas las comparaciones que se corrieron, no solo la ganadora. Con eso, un
lector puede juzgar solo. Cuando las pruebas son muchas y planeadas de antemano, existen correcciones
por comparaciones múltiples (Bonferroni es la más simple: exigir p < alfa dividido por el número de
pruebas), pero eso es de un curso posterior. Lo que sí se exige en este curso es la parte honesta:
**declarar cuántas preguntas le hizo a los datos.**

**Y la formulación que conviene llevarse:** un p-valor solo significa algo si la hipótesis se escribió
**antes** de mirar. Un p-valor obtenido después de veinte intentos no es evidencia, es un sorteo.

</details>

### 6.2 La misma simulación, con alfa = 0,01

Si alfa es la tasa de falsos positivos que uno acepta, bajar de 0,05 a 0,01 tiene que reducir esa tasa a
alrededor de 1 de cada 100. La celda de abajo lo mide sobre el mismo arreglo `p_valores_h0`, donde
sabemos con certeza que no hay nada que encontrar.

In [ ]:
fraccion_p_menor_001 = (p_valores_h0 < 0.01).mean()

print(f'Fracción con p < 0,05 : {(p_valores_h0 < 0.05).mean():.2%}')
print(f'Fracción con p < 0,01 : {fraccion_p_menor_001:.2%}')
print()
print('Exigir más evidencia reduce los falsos positivos. Lo que cuesta no se ve aquí:')
print('con alfa más chico también se dejan pasar más diferencias que sí existen.')

**Pregunta de interpretación 9.** Bajar alfa a 0,01 dejó los falsos positivos por debajo del 1%.
Si es tan barato, ¿por qué el curso no pide alfa = 0,01 siempre, o alfa = 0,001? Nombre lo que se paga,
y dé un ejemplo donde ese precio sea inaceptable.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Lo que se paga es error tipo II: falsos negativos.** Alfa chico exige más evidencia para condenar, y
eso significa soltar a más culpables. Con alfa = 0,001 muchas diferencias reales pero moderadas se
quedan sin detectar, y el informe concluye "no encontramos nada" cuando lo que faltó fue exigencia
calibrada, no efecto.

**Los dos errores no se minimizan a la vez.** Se elige cuál duele más, y esa elección es del dominio, no
de la estadística.

**Ejemplo donde el precio es inaceptable:** un tamizaje temprano de una enfermedad tratable. Un falso
positivo cuesta un examen de confirmación y un susto; un falso negativo cuesta un diagnóstico tardío.
Ahí conviene alfa generoso. Al revés, la física de partículas exige el equivalente a p < 0,0000003 antes
de anunciar una partícula, porque anunciar una que no existe le cuesta la credibilidad a todo el campo.

</details>

### 6.3 Las tres malinterpretaciones

Ahora que vio la simulación, las tres frases erróneas se caen solas.

| Frase que se va a oír | Por qué está mal |
|-----------------------|------------------|
| "p = 0,03, entonces hay 3% de probabilidad de que H0 sea cierta" | El p-valor **asume** que H0 es cierta. No puede devolverle la probabilidad de lo que asumió |
| "p = 0,06, entonces no hay efecto" | Significa que no encontró evidencia suficiente. No es lo mismo que demostrar que no hay nada |
| "p pequeñísimo, entonces el efecto es grande" | El p-valor mezcla tamaño del efecto con tamaño de muestra. Con n grande casi todo sale significativo, y eso es la sección 9 |

**Y sobre el 0,05.** Fisher lo propuso en los años 20 porque le pareció cómodo. No hay nada
matemáticamente especial en ese número. La física de partículas exige 5 sigma (p < 0,0000003) porque
anunciar una partícula que no existe es carísimo; la investigación exploratoria a veces usa 0,10. El
umbral debería depender del costo de equivocarse, y ese costo lo pone el negocio, no la estadística.

Corolario que hay que decir en voz alta: **p = 0,049 y p = 0,051 son, en la práctica, el mismo
resultado.** Caen en lados distintos de una línea arbitraria y nada más.

### 6.4 Los dos errores posibles

|  | H0 es cierta en la realidad | H0 es falsa en la realidad |
|--|------------------------------|----------------------------|
| **Rechazo H0** | **Error tipo I** (falso positivo) — condenar a un inocente | Correcto |
| **No rechazo H0** | Correcto | **Error tipo II** (falso negativo) — soltar a un culpable |

Alfa es la probabilidad que usted acepta de cometer **error tipo I**: son exactamente las 5% de pruebas
que salieron "significativas" en la simulación de arriba sin que hubiera nada.

Bajar alfa de 0,05 a 0,01 reduce el error tipo I y **aumenta** el tipo II. No se pueden minimizar los
dos a la vez; se elige cuál duele más. Un tamizaje de cáncer prefiere el tipo I (asustar a alguien sano,
que luego se descarta) antes que el tipo II (mandar a la casa a alguien enfermo). Un sistema judicial
prefiere lo contrario. **La estadística no decide eso: lo decide quien asume el costo.**

**Pregunta de interpretación 10.** Un informe dice: *"la diferencia no resultó significativa (p = 0,21),
por lo que se concluye que los dos grupos son iguales"*. Hay dos cosas mal en esa frase. Nómbrelas.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Primera: acepta H0.** "Se concluye que son iguales" es el veredicto de inocencia que un tribunal no
puede dar. Lo que muestra p = 0,21 es que **no hay evidencia suficiente** para afirmar que difieren. La
redacción correcta es "no encontramos evidencia de que los grupos difieran (p = 0,21)".

**Segunda: no dice nada del tamaño del efecto ni del n.** Un p de 0,21 con dos grupos de 15 personas y
una diferencia observada de 8 puntos es un estudio que no tenía con qué detectar nada: probablemente
haya un efecto y le faltó muestra. Un p de 0,21 con dos grupos de 50.000 y una diferencia de 0,01
puntos es evidencia bastante buena de que, si hay algo, es diminuto. Son situaciones opuestas y el
p-valor solo no las distingue.

De ahí sale la regla de reporte de la sección 10: **el p-valor nunca va solo.**

</details>

---

## 7. Prueba de una muestra: contra una meta

**La pregunta.** El Ministerio tiene una meta de matriculación del 90%. ¿Nuestros datos son compatibles
con esa meta?

**Paso 1 — Plantear**, antes de mirar nada:

- **H0:** la media poblacional de la tasa de matriculación es 90%.
- **H1:** es distinta de 90%.

**Paso 2 — alfa** = 0,05.

**`stats.ttest_1samp(x, popmean=...)`** compara la media de **un** grupo contra un número fijo de
referencia. El argumento `popmean` es ese número. No confundir con `ttest_ind`, que compara dos grupos
entre sí.

In [ ]:
META = 90.0

t_meta, p_meta = stats.ttest_1samp(matriculacion, popmean=META)

print(f'Media observada : {media:.2f} %')
print(f'Meta            : {META:.2f} %')
print(f'Diferencia      : {media - META:.2f} puntos porcentuales   <- el tamaño del efecto')
print()
print(f'Estadístico t   : {t_meta:.3f}')
print(f'p-valor         : {p_meta:.2e}')
print()
if p_meta < 0.05:
    print('Paso 4: p < 0,05  ->  RECHAZAMOS H0.')
    print('La tasa está sistemáticamente por debajo de la meta, y no por azar de muestreo.')
else:
    print('Paso 4: p >= 0,05 ->  NO rechazamos H0.')

**Léalo con la definición en la mano.** El p-valor es del orden de 10 elevado a -19: *si la media real
fuera exactamente 90%, ver unos datos como estos sería extraordinariamente improbable.* Por eso
rechazamos H0.

Y note el orden en que están impresos los números: **la diferencia de 4,47 puntos porcentuales va
antes que el p-valor**. Esa es la que le sirve al Ministerio para decidir. El p-valor solo dice que no
es casualidad.

**Pregunta de interpretación 11.** Escriba, en una sola frase y en lenguaje de Ministerio, la
conclusión de esta prueba. Después revise su frase contra estas dos trampas: ¿dijo "se demostró que la
meta no se cumple"? ¿Puso el p-valor antes que la diferencia? Corrija lo que haga falta y explique por
qué era un problema.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Una redacción que aguanta:** *"La tasa de matriculación observada es de 85,53%, es decir 4,47 puntos
porcentuales por debajo de la meta del 90%. La diferencia no se explica por el azar de muestreo
(t = -9,23; p < 0,001; n = 462)."*

**Trampa 1, "se demostró".** Una prueba no demuestra: aporta evidencia. Y con datos observacionales
tampoco explica por qué. La frase segura describe la brecha y dice que no es casualidad; no promete
certeza ni causa.

**Trampa 2, el orden.** El p-valor de 10 elevado a -19 no es la noticia: la noticia son los 4,47 puntos.
Si el p-valor abre la frase, quien la lee se queda con "significativo" y sin idea del tamaño de la
brecha, que es exactamente el número con el que se decide un presupuesto. La regla del curso es
tamaño del efecto primero, precisión después, p-valor de último.

**Un detalle extra que separa a un analista de un ejecutor de código:** esa media junta 33 departamentos
y 14 años en un solo número. Es correcta y es poco útil sola; la pregunta siguiente, la que se espera en
la sustentación, es si la brecha se está cerrando con el tiempo o si hay departamentos que la sostienen.

</details>

---

## 8. Prueba de dos muestras: urbanizado contra rural disperso

**La pregunta.** ¿La deserción escolar es distinta en los departamentos urbanizados y en los rurales
dispersos?

**Paso 1 — Plantear:**

- **H0:** la deserción media es igual en los dos grupos.
- **H1:** la deserción media es distinta entre los dos grupos.

**Paso 2 — alfa** = 0,05.

### Regla de oro: descriptivos primero, prueba después

Nunca se corre un t-test sin haber mirado antes las medias, las desviaciones y los tamaños de cada
grupo. Si no puede explicar el resultado con los descriptivos, el p-valor no lo va a salvar. Y al revés:
si los descriptivos ya cuentan la historia, el p-valor solo confirma que no es casualidad.

In [ ]:
resumen = pd.DataFrame({
    'grupo': ['Urbanizado', 'Rural disperso'],
    'n': [len(urbano), len(rural)],
    'media': [urbano.mean(), rural.mean()],
    'desv_est': [urbano.std(ddof=1), rural.std(ddof=1)],
})
print(resumen.to_string(index=False))

diferencia = urbano.mean() - rural.mean()
print()
print(f'TAMAÑO DEL EFECTO: {diferencia:.2f} puntos porcentuales')
print('Este es el número que le importa al Ministerio. El p-valor viene después.')

### `stats.ttest_ind` y el argumento `equal_var`

| Argumento | Qué es |
|-----------|--------|
| primero, segundo | Las dos series a comparar |
| `equal_var` | Si asumimos que los dos grupos tienen la misma varianza |

**Regla del curso: `equal_var=False` siempre.** Eso activa el **t-test de Welch**, que no asume
varianzas iguales. Mire las desviaciones de la tabla de arriba: 1,15 frente a 2,00. Asumir que son
iguales sería sencillamente falso.

Cuando las varianzas sí son parecidas, Welch da prácticamente el mismo resultado. Cuando no lo son,
Welch es el correcto y el clásico está mal. Por eso se pone siempre: no cuesta nada y evita un supuesto
que casi nunca se cumple con datos reales.

In [ ]:
t_welch, p_welch = stats.ttest_ind(urbano, rural, equal_var=False)

print(f'Estadístico t : {t_welch:.3f}')
print(f'p-valor       : {p_welch:.2e}')
print()
if p_welch < 0.05:
    print('Paso 4: p < 0,05  ->  RECHAZAMOS H0.')
    print('Las medias difieren más de lo que explicaría el azar de muestreo.')
else:
    print('Paso 4: p >= 0,05 ->  NO rechazamos H0.')

### 8.2 Welch contra el clásico, uno al lado del otro

La regla del curso no es un capricho, y la forma de convencerse es correr las dos. Sin `equal_var=False`
la función usa el t-test clásico, que asume varianzas iguales; las de la tabla de arriba no lo son.

In [ ]:
t_clasico, p_clasico = stats.ttest_ind(urbano, rural)

print(f'Welch   : t = {t_welch:.3f}   p = {p_welch:.2e}')
print(f'Clásico : t = {t_clasico:.3f}   p = {p_clasico:.2e}')
print()
print('Las desviaciones eran distintas y aun así la conclusión no se movió.')

**Pregunta de interpretación 12.** El número cambió y la conclusión no. Entonces: ¿por qué la regla del
curso sigue siendo usar Welch **siempre**, si en este caso daba igual? Y una segunda, que es la que
importa: ¿cómo habría sabido usted, **antes** de correr las dos, que aquí daba igual?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Porque una regla que hay que decidir cada vez es una regla que algún día se decide mal.** Welch no
cuesta nada: cuando las varianzas se parecen da prácticamente lo mismo que el clásico, y cuando no se
parecen es el correcto. Una opción por defecto que nunca empeora y a veces salva es una opción por
defecto sensata. El clásico solo tiene sentido cuando hay una razón positiva para afirmar varianzas
iguales, y con datos reales esa razón casi nunca existe.

**Y aquí daba igual porque el p-valor estaba lejísimos del umbral.** La diferencia entre los dos
métodos mueve el estadístico y los grados de libertad un poco; cuando el resultado no está pegado a
0,05, ese "poco" no alcanza a voltear nada. El caso peligroso es el contrario: un p de 0,048 con el
clásico y 0,061 con Welch. Ahí la elección del método decide el veredicto, y quien la haya tomado
después de ver los dos números ya no está haciendo estadística.

**Corolario práctico:** cuando un resultado depende de qué variante de la prueba se usó, el resultado no
es sólido. Eso se reporta, no se esconde.

</details>

### 8.3 El intervalo de confianza **de la diferencia**

El p-valor dice *"sí hay diferencia"*. El intervalo dice **de cuánto**, y es el número que cierra un
informe.

Lo calculamos con la fórmula de Welch, a mano, para que se vea que no hay magia:

- El **error estándar de la diferencia** combina los dos errores estándar: `raíz(v1/n1 + v2/n2)`.
- Los **grados de libertad de Welch** salen de la fórmula de Welch-Satterthwaite, que es fea y no hay
  que memorizar. Lo que sí hay que saber es que no son `n1 + n2 - 2`.

Y después, la misma `stats.t.interval` de siempre, con la diferencia en `loc` y ese error estándar en
`scale`.

In [ ]:
n1, n2 = len(urbano), len(rural)
v1, v2 = urbano.var(ddof=1), rural.var(ddof=1)

ee_dif = np.sqrt(v1 / n1 + v2 / n2)
gl = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))

ic_dif = stats.t.interval(0.95, df=gl, loc=diferencia, scale=ee_dif)

print(f'Diferencia de medias    : {diferencia:.2f} puntos porcentuales')
print(f'Error estándar          : {ee_dif:.4f}')
print(f'Grados de libertad      : {gl:.1f}')
print(f'IC 95% de la diferencia : ({ic_dif[0]:.2f} , {ic_dif[1]:.2f})')
print()
print('Lectura: el método dice que la ventaja de los urbanizados está entre')
print(f'{abs(ic_dif[1]):.2f} y {abs(ic_dif[0]):.2f} puntos porcentuales.')
print('Que el intervalo NO contenga el cero es la otra cara del p < 0,05.')

**Pregunta de interpretación 13.** Tiene dos resultados sobre la misma comparación: un p-valor por
debajo de 0,001 y un intervalo de la diferencia. Si solo pudiera poner **uno** de los dos en la primera
línea del informe al Ministerio, ¿cuál pone y por qué? Y explique qué información le da el intervalo
que el p-valor no le da.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Va el intervalo.** El p-valor responde una pregunta de sí o no: ¿esto se explica por azar? El
intervalo responde la que se usa para decidir: **¿de cuánto es la brecha, y con qué precisión la
medimos?** Un ministerio no asigna presupuesto con un "sí"; lo asigna con "entre tanto y tanto puntos
porcentuales".

**Lo que el intervalo dice de más:**

1. **El tamaño del efecto**, en las unidades del problema, que es lo único que permite juzgar si
   importa.
2. **La precisión**, en su ancho. Un intervalo angosto es una medición firme; uno ancho avisa que con
   estos datos no se puede afinar más.
3. **El veredicto del p-valor, de regalo.** Que no contenga el cero es exactamente lo mismo que
   p < 0,05. El intervalo contiene la decisión y además el número; el p-valor solo contiene la decisión.

**Por eso la plantilla de la sección 10 los pide juntos y en ese orden.** El p-valor no se esconde: se
pone donde le corresponde, al final del paréntesis.

</details>

### 8.4 El gráfico de la comparación

Las reglas de la clase 8 siguen vigentes: el gráfico tiene que poder leerse solo, sin que nadie lo
explique al lado. Título con el hallazgo, ejes rotulados con unidad, el `n` de cada grupo visible y la
anotación del resultado sobre el dibujo.

In [ ]:
fig, eje = plt.subplots(figsize=(8, 5))

cajas = eje.boxplot([urbano, rural],
                    tick_labels=[f'Urbanizado\n(n={n1})', f'Rural disperso\n(n={n2})'],
                    patch_artist=True, widths=0.5)
for parche, color in zip(cajas['boxes'], ['#2980b9', '#c0392b']):
    parche.set_facecolor(color)
    parche.set_alpha(0.65)

eje.set_xlabel('Grupo territorial (proxy construido a partir del código DANE)')
eje.set_ylabel('Tasa de deserción escolar (%)')
eje.set_title(f'La deserción es {abs(diferencia):.2f} puntos porcentuales menor en los\n'
              'departamentos urbanizados que en los rurales dispersos')

alto = max(urbano.max(), rural.max())
eje.plot([1, 1, 2, 2], [alto + 0.4, alto + 0.9, alto + 0.9, alto + 0.4], color='black', lw=1.2)
eje.text(1.5, alto + 1.0, 'p < 0,001', ha='center', fontsize=11)
eje.set_ylim(top=alto + 2.0)

plt.tight_layout()
plt.show()

---

## 9. Por qué "significativo" no quiere decir "importante"

### Para entender qué está pasando · La confusión más cara del oficio

**Significativo** es una palabra que en estadística no quiere decir lo que quiere decir en español. No
significa "importante", ni "grande", ni "relevante". Significa: **más grande de lo que el azar de
muestreo explicaría con facilidad.**

Y el p-valor mezcla dos cosas distintas en un solo número:

- **El tamaño del efecto**: qué tan grande es la diferencia.
- **El tamaño de la muestra**: con cuánta precisión la midió.

Con n suficientemente grande, **una diferencia trivial sale significativa**. No es un defecto: es la
definición funcionando. Más datos significa más precisión, y más precisión permite detectar
diferencias más pequeñas, incluso las que a nadie le importan.

**La simulación.** La celda de abajo fabrica dos grupos que difieren de verdad en **0,3 puntos
porcentuales** de aprobación. Tres décimas: nada que le cambie la vida a ningún estudiante. Después
corre la misma prueba con muestras cada vez más grandes.

Mire cómo el p-valor se derrumba mientras la diferencia se queda donde estaba.

In [ ]:
DIFERENCIA_REAL = 0.3        # tres décimas de punto porcentual: trivial
DESVIACION = 4.5
TAMANOS = [50, 200, 1000, 5000, 20000, 100000]

generador_efecto = np.random.default_rng(1305)
grupo_grande_a = generador_efecto.normal(90.0, DESVIACION, max(TAMANOS))
grupo_grande_b = generador_efecto.normal(90.0 + DIFERENCIA_REAL, DESVIACION, max(TAMANOS))

filas = []
for tamano in TAMANOS:
    a = grupo_grande_a[:tamano]
    b = grupo_grande_b[:tamano]
    _, p = stats.ttest_ind(b, a, equal_var=False)
    filas.append({'n_por_grupo': tamano,
                  'diferencia_observada': round(b.mean() - a.mean(), 3),
                  'p_valor': p,
                  'significativo': 'SÍ' if p < 0.05 else 'no'})

tabla_efecto = pd.DataFrame(filas)
print(tabla_efecto.to_string(index=False,
                             formatters={'p_valor': lambda v: f'{v:.2e}'}))
print()
print('La diferencia real es SIEMPRE la misma: 0,3 puntos porcentuales.')
print('Lo único que cambia es cuánta gente se midió. Y el veredicto se voltea.')

### 9.2 El punto donde se voltea

La celda de abajo no calcula nada nuevo: **lee la tabla** y se queda con el primer tamaño de grupo cuyo
p-valor cae por debajo de 0,05. Ese número es la respuesta a la pregunta incómoda: *¿cuántos datos hacen
falta para que una diferencia que no le importa a nadie aparezca como un hallazgo?*

In [ ]:
significativos = tabla_efecto[tabla_efecto['p_valor'] < 0.05]
n_minimo_significativo = int(significativos['n_por_grupo'].iloc[0])

print(f'A partir de n = {n_minimo_significativo} por grupo, una diferencia real de '
      f'{DIFERENCIA_REAL} puntos')
print('porcentuales pasa a ser "estadísticamente significativa".')
print()
print('La diferencia nunca cambió. Cambió el presupuesto del estudio.')

**Pregunta de interpretación 14.** Con la tabla y ese número a la vista, responda dos cosas. Primero:
¿por qué una diferencia de 0,3 puntos porcentuales es "no significativa" con n = 200 y "significativa"
con n = 100.000, si la diferencia real es idéntica en los dos casos? Segundo: al revés. Su equipo compara
dos grupos de 40 departamentos y le da p = 0,30. ¿Puede concluir que no hay diferencia?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Primero.** Porque el p-valor no mide el tamaño de la diferencia: mide qué tan improbable sería verla
si no existiera, y eso depende de la precisión con que la midió. Más n significa error estándar más
chico, y con un error estándar suficientemente chico hasta una diferencia diminuta queda fuera del rango
de lo que el azar explica con facilidad. No es un defecto de la prueba: es la prueba haciendo
exactamente lo que promete.

**Segundo, y es el error simétrico.** No. Con n = 40 por grupo, el estudio probablemente **no tenía con
qué** detectar una diferencia moderada: el intervalo de la diferencia va a ser ancho y va a contener
tanto el cero como valores que sí importarían. La conclusión honesta es "con esta muestra no podemos
distinguir una diferencia de cero", que no es lo mismo que "no hay diferencia".

**La manera de no equivocarse en ninguno de los dos sentidos es la misma:** mirar el intervalo de la
diferencia. Un intervalo angosto alrededor de cero sí dice "si hay algo, es diminuto". Uno ancho dice
"no sabemos". El p-valor solo no distingue esos dos casos, y por eso nunca va solo.

</details>

**Pregunta de interpretación 15.** Con esa tabla a la vista: si mañana alguien le presenta un informe que dice
*"encontramos una diferencia estadísticamente significativa (p < 0,001)"* y no dice nada más, ¿qué dos
preguntas hace usted antes de creerle?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**1. ¿De cuánto es la diferencia, en unidades que yo entienda?** Puntos porcentuales, pesos, metros
cúbicos, estudiantes. Si la respuesta es "0,3 puntos", el hallazgo es real y no le sirve a nadie.

**2. ¿Cuál es el n?** Con n grande, el p-valor pequeño era predecible antes de mirar los datos. Con n
chico, un p pequeño con un efecto grande es mucho más informativo.

Una tercera, que vale la pena tener a mano: **¿cuál es el intervalo de confianza del efecto?** Responde
las dos anteriores de un solo golpe y por eso es la que se pide en el formato de reporte.

Y la formulación corta, que sirve para el resto de su vida profesional: **la significancia estadística
responde "¿es real?"; el tamaño del efecto responde "¿me importa?".** Son dos preguntas distintas y
hacen falta las dos.

</details>

---

## 10. Cómo se reporta un resultado

Esta es la parte que se le va a exigir el resto del semestre, incluido el Momento 3.

### Los cuatro elementos obligatorios, en este orden

1. **Tamaño del efecto** en unidades del negocio (puntos porcentuales, pesos, m3). No en unidades
   estadísticas.
2. **Intervalo de confianza** de ese efecto.
3. **P-valor** y tamaños de muestra.
4. **Una frase en lenguaje llano** que un funcionario pueda leer sin saber qué es un t-test.

### El reporte de hoy

> Los departamentos con área metropolitana grande registran una deserción escolar **1,55 puntos
> porcentuales menor** que los departamentos de Amazonía, Orinoquía y Chocó (3,57% frente a 5,12%).
> La diferencia es estadísticamente significativa (IC 95% de la diferencia: [-1,97, -1,14]; t = -7,39;
> p < 0,001; n = 139 y n = 116). La agrupación territorial es un proxy construido a partir del código
> DANE, no una columna del dataset.

### Tres contraejemplos que se rechazan

| Lo que se escribe | Por qué no sirve |
|-------------------|------------------|
| "p < 0,05, entonces sí hay diferencia" | No dice de cuánto. Inútil para decidir |
| "La deserción rural es mayor" | Sin número y sin incertidumbre. Es una opinión, no un hallazgo |
| "Se demostró que la ruralidad causa deserción" | Esto es observacional. Correlación, no causalidad. Se dice "se asocia con" |

### Reglas de formato

- **`p < 0,001`** se reporta así, nunca como `p = 0.0000000000055`. Por encima de 0,001, tres decimales.
- El `n` va **siempre**.
- Toda decisión metodológica que usted haya tomado se declara: qué se filtró, qué grupo se construyó,
  qué filas se botaron.

La celda de abajo arma el reporte automáticamente. No es para memorizarla: es para que vea que el
formato es una plantilla, y que una vez fijado, cumplirlo no cuesta trabajo.

In [ ]:
def reportar_diferencia(nombre_a, a, nombre_b, b, unidad='puntos porcentuales'):
    """Arma el reporte de una comparación de dos grupos con los cuatro elementos obligatorios."""
    dif = a.mean() - b.mean()
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    ee = np.sqrt(va / na + vb / nb)
    grados = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    ic = stats.t.interval(0.95, df=grados, loc=dif, scale=ee)
    t, p = stats.ttest_ind(a, b, equal_var=False)
    p_texto = 'p < 0,001' if p < 0.001 else f'p = {p:.3f}'
    veredicto = 'es' if p < 0.05 else 'NO es'
    return (
        f'{nombre_a} registra {abs(dif):.2f} {unidad} '
        f"{'más' if dif > 0 else 'menos'} que {nombre_b} "
        f'({a.mean():.2f} frente a {b.mean():.2f}). '
        f'La diferencia {veredicto} estadísticamente significativa '
        f'(IC 95% de la diferencia: [{ic[0]:.2f}, {ic[1]:.2f}]; '
        f't = {t:.2f}; {p_texto}; n = {na} y n = {nb}).'
    )


print(reportar_diferencia('El grupo urbanizado', urbano, 'el rural disperso', rural))

---

## 11. Preguntas que siempre salen

**¿Por qué asumimos que H0 es cierta si lo que queremos es refutarla?**
Es una demostración por contradicción, la misma lógica de las matemáticas: se asume la explicación
aburrida y se pregunta si los datos son compatibles con ella. Si bajo H0 los datos serían rarísimos, hay
evidencia en contra de H0. Se necesita un supuesto de partida para poder medir qué tan raro es "raro".

**Si p = 0,04, ¿puedo decir que hay 96% de probabilidad de que mi hipótesis sea cierta?**
No, y es la malinterpretación más cara. El p-valor **parte** de suponer que H0 es cierta; no puede
devolverle la probabilidad de que H0 sea cierta. Para eso hace falta estadística bayesiana, que no se ve
en este curso. Lo único que dice p = 0,04 es: *si H0 fuera cierta, datos así de extremos aparecerían el
4% de las veces.*

**¿Qué hago si mi p-valor da exactamente 0,05? ¿O 0,051?**
Reportar el valor exacto y ser transparente. La respuesta profesional es: *"el resultado está en el
límite; el tamaño del efecto es X y el intervalo de confianza es [a, b]; con esta muestra no podemos
concluir de forma tajante"*. Nunca "casi significativo" como excusa para concluir lo que uno quería.

**¿Cuál es la diferencia entre desviación estándar y error estándar?**
La desviación estándar mide qué tan dispersos están los **datos**. El error estándar mide qué tan
dispersa estaría la **media** si repitiera el estudio. El segundo es el primero dividido por la raíz de
n, así que siempre es más pequeño y se achica al recolectar más datos. Confundirlos produce intervalos
absurdamente anchos, y es la sección 5.2.

**¿Y si mis datos no son normales?**
El t-test es bastante robusto cuando n es grande, precisamente por el teorema del límite central de la
sección 2.4: lo que tiene que ser normal es la distribución de la **media**, no la de los datos. Con
muestras pequeñas y distribuciones muy asimétricas, la alternativa es una prueba no paramétrica como
`stats.mannwhitneyu`, que compara medianas y no exige normalidad. Está en la parte opcional del reto.

**¿Cuándo uso `ttest_ind` y cuándo `ttest_1samp`?**
`ttest_1samp` compara un grupo contra un número fijo de referencia (una meta, un estándar, un valor
histórico). `ttest_ind` compara dos grupos independientes entre sí. Si los mismos sujetos se miden dos
veces, antes y después, no es ninguna de las dos: es `ttest_rel`, y es la sección 5.3.

**Mi prueba dio no significativa. ¿Puedo cambiar los grupos para que dé?**
No, y tiene nombre: **p-hacking**. Probar variantes hasta que una dé p < 0,05 garantiza encontrar algo
por puro azar: la simulación de la sección 6 lo muestra con 2.000 pruebas sobre datos sin ninguna
diferencia, de las cuales alrededor de 100 salieron "significativas". La regla es plantear la hipótesis
**antes** y reportar todo lo que se probó, no solo lo que salió bonito.

**Si el t-test sale significativo, ¿ya demostré que una cosa causa la otra?**
No. Estos son datos observacionales: nadie asignó al azar los departamentos a ser urbanos o rurales. Un
t-test significativo dice que las medias difieren más de lo que explicaría el azar, y nada más. La frase
segura es "se asocia con", nunca "causa". Es la clase 5, sección 8, otra vez.

**¿Este dataset sirve para el proyecto de mi equipo?**
No. Los CSV de las clases son material de enseñanza, elegidos por lo que permiten enseñar. El dataset
del proyecto lo consigue cada equipo, de la fuente que quiera, y tiene que poder decir de dónde salió y
bajo qué condiciones se puede usar.

---

## Resumen

| Lo que hizo | Con qué |
|-------------|---------|
| Error estándar de la media | `stats.sem(x)`, o `x.std(ddof=1) / np.sqrt(len(x))` |
| Intervalo de confianza de una media | `stats.t.interval(0.95, df=n-1, loc=media, scale=error_estandar)` |
| Comparar una media contra una meta | `stats.ttest_1samp(x, popmean=meta)` |
| Comparar dos grupos independientes | `stats.ttest_ind(a, b, equal_var=False)` |
| Intervalo de la diferencia (Welch) | `stats.t.interval(0.95, df=gl_welch, loc=dif, scale=ee_dif)` |
| Simular muchos estudios | `np.random.default_rng(semilla).choice(valores, n, replace=True)` |

**Las siete reglas que no se negocian:**

1. `dropna()` **antes** de calcular, siempre, y se reporta el `n` que quedó.
2. En `scale` va el **error estándar**, no la desviación estándar.
3. El 95% describe la **tasa de acierto del método**, no la certeza sobre este intervalo.
4. Nunca se acepta H0: se **falla en rechazarla**.
5. El p-valor es la probabilidad de los datos **si H0 fuera cierta**. Nada más.
6. 0,05 es una convención, y 0,049 y 0,051 son el mismo resultado.
7. Un reporte sin tamaño del efecto no es un reporte.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo explicar por qué el error estándar es más chico que la desviación estándar, y qué lo achica.
- [ ] Puedo enunciar qué significa el 95% de un intervalo **y** qué no significa, sin leer.
- [ ] Puedo escribir H0 y H1 para una pregunta nueva, en una línea cada una, antes de ver los datos.
- [ ] Puedo definir el p-valor con una frase que empiece por "la probabilidad de observar datos así de
      extremos si H0 fuera cierta".
- [ ] Puedo decir por qué un p < 0,001 con n = 200.000 puede no importarle a nadie.

**Siguiente:** bloque 3, el reto. Mismo archivo, **cinco variables que no aparecieron aquí**, y una
comparación temporal que no se hizo hoy.